In [1]:
import pandas as pd
df_exp=pd.read_csv("C:/나이스/summary_experience_dataset.csv")
df_new=pd.read_csv("C:/나이스/summary_new_dataset.csv")

df_combined = pd.concat([df_exp['question_text'], df_new['question_text']], ignore_index=True)

# 결과를 새로운 DataFrame으로 만들기
df_questions = pd.DataFrame({'question_text': df_combined})
# 결과 확인
print(df_questions.head())
df_questions.to_csv('questions.csv')

                                       question_text
0                        디자이너로서 앞으로의 목표에 관해서 설명해 주세요
1  협업을 할 때 사교성이 좋은 편인지 궁금합니다 그리고 또 사교성을 키우기 위해 어떤...
2  지원자님이 태어나서 지금까지 한 일들 가운데 가장 후회했던 일이 무엇인가요 한 가지...
3  대학 생활을 보내면서 가장 힘들었던 경험은 무엇인가요 그것을 어떻게 극복하였는지 예...
4                                  가장 자신있는 작업은 무엇일까요


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from konlpy.tag import Okt
import re
import random
import numpy as np

from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()

okt = Okt()


In [2]:
def clean_text(text):
    text = re.sub(r"[^가-힣\s]", "", text)
    return text.strip()

def tokenize_korean_text(text):
    return ' '.join(okt.nouns(clean_text(text)))


In [3]:
def get_clustered_dataframe(df_source, num_clusters=7):
    if df_source.empty or 'question_text' not in df_source.columns:
        print("오류: 입력 DataFrame이 비어있거나 'question_text' 컬럼이 없습니다.")
        return pd.DataFrame()

    df_copy = df_source.copy()
    df_copy['processed_question'] = df_copy['question_text'].apply(tokenize_korean_text)

    vectorizer = TfidfVectorizer(max_features=2000, min_df=5, max_df=0.8)
    X = vectorizer.fit_transform(df_copy['processed_question'])

    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df_copy['cluster'] = kmeans.fit_predict(X)

    return df_copy


In [4]:
def generate_new_questions_for_cluster(cluster_id, clustered_df, llm_model, num_examples=5, num_to_generate=3):
    cluster_questions = clustered_df[clustered_df['cluster'] == cluster_id]['question_text'].tolist()

    if not cluster_questions:
        print(f"클러스터 {cluster_id}에 질문이 없습니다.")
        return []

    selected_examples = random.sample(cluster_questions, min(len(cluster_questions), num_examples))
    examples_str = "\n- ".join(selected_examples)

    template = """
    당신은 면접 질문 생성 전문가입니다.
    아래는 특정 유형의 면접 질문 예시입니다. 이 예시들의 주제, 스타일, 질문의 의도 등을 고려하여,
    이와 유사한 새로운 면접 질문을 {num_to_generate}개 생성해 주세요.
    각 질문은 한 줄로 작성해 주시고, 번호를 붙여주세요.

    ---
    [면접 질문 예시]
    - {examples}
    ---

    [새로운 질문 생성]
    """
    prompt = PromptTemplate(input_variables=["examples", "num_to_generate"], template=template)
    chain = LLMChain(llm=llm_model, prompt=prompt)

    print(f"\n--- 클러스터 {cluster_id}에 대한 새로운 질문 생성 중 ---")
    print(f"  (예시 질문: {selected_examples[:3]}...)")

    try:
        response = chain.invoke({
            "examples": examples_str,
            "num_to_generate": num_to_generate
        })

        if isinstance(response, dict) and "text" in response:
            generated_text = response["text"].strip()
        elif hasattr(response, "content"):
            generated_text = response.content.strip()
        else:
            generated_text = str(response).strip()

        new_questions = [q.strip() for q in generated_text.split('\n') if q.strip()]
        return new_questions

    except Exception as e:
        print(f"LLM 질문 생성 중 오류 발생: {e}")
        return []


In [5]:
# CSV 파일 로드
csv_file_path = 'questions.csv'
try:
    df_questions = pd.read_csv(csv_file_path)
    if 'question_text' not in df_questions.columns:
        print("오류: CSV 파일에 'question_text' 컬럼이 없습니다.")
        exit()
    print(f"'{csv_file_path}'에서 {len(df_questions)}개 질문을 로드했습니다.")
except Exception as e:
    print(f"CSV 파일 로드 실패: {e}")
    exit()

# 군집화 실행
num_clusters_to_use = 7
print(f"\n{num_clusters_to_use}개 클러스터로 질문 군집화 중...")
clustered_data_df = get_clustered_dataframe(df_questions, num_clusters=num_clusters_to_use)


'questions.csv'에서 68074개 질문을 로드했습니다.

7개 클러스터로 질문 군집화 중...


In [8]:
if not clustered_data_df.empty:
    if os.getenv("OPENAI_API_KEY") is None:
        print("OPENAI_API_KEY가 설정되지 않았습니다. 질문 생성을 건너뜁니다.")
    else:
        llm = ChatOpenAI(model="gpt-4o", temperature=0.7, max_tokens=500)

        all_generated_questions = {}
        for cluster_id in range(num_clusters_to_use):
            generated_q_list = generate_new_questions_for_cluster(
                cluster_id=cluster_id,
                clustered_df=clustered_data_df,
                llm_model=llm,
                num_examples=1,
                num_to_generate=1
            )
            if generated_q_list:
                all_generated_questions[f'Cluster_{cluster_id}'] = generated_q_list

        print("\n\n=== 생성된 질문 모음 ===")
        for cluster_name, questions in all_generated_questions.items():
            print(f"\n[{cluster_name}]")
            for q in questions:
                print(f"- {q}")
else:
    print("군집화된 데이터가 없습니다.")



--- 클러스터 0에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['어 직무를 수행하다가 모르는 과제가 등장할 수 있습니다 그런 경우에는 어떻게 해결하시겠습니까 그리고 그렇게 해결하시는 해결법과 이유를 말씀해 주시길 바랍니다']...)

--- 클러스터 1에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['지원자님께서 지금까지 협업을 하면서 가장 어려웠던 점은 무엇이었습니까 그리고 그 어려움을 어떻게 해결했는지 함께 말씀해 주세요']...)

--- 클러스터 2에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['직무 지식 역량 태도 이 세 가지 요소 중 가장 중요하다고 생각하는 것과 그 이유를 말씀 부탁드리겠습니다']...)

--- 클러스터 3에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['개인적으로 공부를 어떻게 하시나요 강의나 책을 활용한 공부를 말씀해 주셔도 좋으니까요 본인만의 공부 방법에 대해 말씀해 주시길 바랍니다']...)

--- 클러스터 4에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['입사 후 쉬운 일 어려운 일이 있을 때 이 일에 대한 분배를 어떻게 하실 건지 저희에게 이야기 부탁드리겠습니다 어떻게 나눠서 처리할 생각이신지 저희에게 이야기 해 주시기 바랍니다']...)

--- 클러스터 5에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['본인만의 스트레스 해소 방법이 있다면 어떤 이유로 그런 방법으로 스트레스를 해소하는지도 말씀해 주세요']...)

--- 클러스터 6에 대한 새로운 질문 생성 중 ---
  (예시 질문: ['글로벌 인재에게 필수인 요소는 무엇이고 면접자님께서 그것을 위해 노력했던 것은 무엇이었습니까']...)


=== 생성된 질문 모음 ===

[Cluster_0]
- 1. 예상치 못한 문제가 발생했을 때 어떻게 대처하시며, 그 과정을 통해 얻은 교훈을 설명해 주시기 바랍니다.

[Cluster_1]
- 1. 지원자님께서 과거 프로젝트에서 예상치 못한 문제가 발생했을